# ResNet50 U-Net with the DEM

Same as `unet_pretrained.ipynb` (phase 1 + augmentation), but the model also gets the elevation (DEM) as a 4th input channel. ResNet's first layer gets an extra DEM input that starts at zero, so training starts from the RGB model and the DEM weights are learned. Weights are saved to `unet_dem.weights.h5`.

In [ ]:
import tensorflow as tf
from tensorflow import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
MAX_UNKNOWN = 1.0 # 1.0 = keep every training tile try 0.5 to drop tiles

In [ ]:
import sys
sys.path.append("../scripts")
import dataset
from dataset import get_tiles, SPLITS
# This notebook's experiment: give the model the elevation twice, in metres AND scaled per image.
# Metres did better on hilly images, scaled did better on flat ones (Ida); with both it can learn when to use which.
dataset.DEM_STANDARDIZE = "both"  # X now has 5 channels: R, G, B, DEM in metres, DEM scaled
X_train, y_train = get_tiles(SPLITS["train"], stride=128,
                             max_unknown=MAX_UNKNOWN)

# Val and test no overlap, keep every tile (the defaults)
X_val, y_val = get_tiles(SPLITS["val"])
X_test, y_test = get_tiles(SPLITS["test"])

In [ ]:
import numpy as np

for name, X, y in [("Train", X_train, y_train), ("Val", X_val, y_val), ("Test", X_test, y_test)]:
    unknown = (y == -1).mean()
    known = y[y != -1]
    print(f"{name:5}  X {X.shape}  y {y.shape}  unknown {unknown:.1%}  flood (of known) {known.mean():.1%}")

In [ ]:
#make copy of the training tiles that keeps only the rgb channels and leaves out the dem part

X_train_rgb = X_train[..., :3]
X_val_rgb = X_val[..., :3]
X_test_rgb = X_test[..., :3]

print(X_train_rgb.shape, X_val_rgb.shape, X_test_rgb.shape)
print("RGB min:", X_train_rgb.min(), "max:", X_train_rgb.max())

## Prepare the images for ResNet50

RGB: same as before (0-255, then `preprocess_input`).
DEM: already centered per image by `dataset.py`. Its spread is only ~1-4 m, so it is multiplied by `DEM_SCALE` to be on a similar scale to the preprocessed RGB.

In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input

DEM_SCALE = np.float32([20, 40])  # metres channel x20, scaled channel x40 (the values each was trained with before)  # was: # the DEM now has std 1 per image (dataset.DEM_STANDARDIZE); x40 keeps it near its old typical spread (about 2 m x 20)

def preprocess(x):
    rgb = preprocess_input(x[..., :3] * 255.0)  # 0-1 to 0-255
    return np.concatenate([rgb, x[..., 3:] * DEM_SCALE], axis=-1)

# in place, a chunk at a time: a second full copy of the training tiles froze the 16 GB PC
X_train_in = X_train
for i in range(0, len(X_train), 256):
    X_train[i:i + 256] = preprocess(X_train[i:i + 256])
X_val_in = preprocess(X_val)
X_test_in = preprocess(X_test)

print(X_train_in.shape)
print("RGB min/max:", X_train_in[..., :3].min(), X_train_in[..., :3].max())
print("DEM (metres) min/max:", X_train_in[..., 3].min(), X_train_in[..., 3].max())
print("DEM (scaled) min/max:", X_train_in[..., 4].min(), X_train_in[..., 4].max())

## Prepare the Labels

In [ ]:
def prepare_labels(y):
    return y.astype("float32")[..., None] # Changes (N, 256, 256) to (N, 256, 256, 1)

y_train_in = prepare_labels(y_train)
y_val_in = prepare_labels(y_val)
y_test_in = prepare_labels(y_test)

print(y_train_in.shape)
print("values:", np.unique(y_train_in))

## Loss function (skips unknown pixels)

In [ ]:
def masked_bce(y_true, y_pred):
    known = tf.cast(y_true != -1, "float32") # 1 = labeled pixel, 0 = unknown, turns T/F into 1.0/0.0
    y_true = tf.where(y_true == -1, 0.0, y_true) # -1 would break the loss , so set iti to 0
                                                 # replaces every -1 with 0. BCE only understands labels of 0 or 1, and a -1 would give nonsense.
    loss = keras.losses.binary_crossentropy(y_true, y_pred)[..., None] #computes loss for every pixel
 
    return tf.reduce_sum(loss * known) / tf.maximum(tf.reduce_sum(known), 1.0) #average over labeled picels only
                                                              #loss * known zeros out the pentalty on unknown pixels
                                                              #tf.reduce_sum / tf.reduce_sum adds up the remaining penalties and divides by how many pixels are known.
                                                              #gives the average penallty over labeled pixels only



In [ ]:
y_true = tf.constant([[[[-1.0], [1.0]], [[0.0], [1.0]]]])   # unknown, flood, dry, flood
y_pred = tf.constant([[[[0.9], [0.9]], [[0.1], [0.8]]]])    # model's guesses
print(masked_bce(y_true, y_pred).numpy())

In [ ]:
# Elevation-guided loss from the EvaNet paper (Sami et al. 2024, section 3.1, equations 2-4 and 7).
# The rule it teaches: water runs downhill. Next to a flooded pixel, anything at the same height or lower
# should be flood. Next to a dry pixel, anything at the same height or higher should be dry.
EVA_LAMBDA = 1.0  # how much the elevation rule counts. 1.0 = the paper's "L_CE + L_eva". 0 = the old loss (plain masked BCE).

def eva_loss(y_true, y_pred):
    # y_true has 2 channels here: our label (-1 unknown, 0 dry, 1 flood) and the elevation.
    # Only "higher or lower than the neighbour" is used, so the scaled DEM the model sees works as it is.
    label, h = y_true[..., :1], y_true[..., 1:]
    gt = tf.where(label == 1, 1.0, tf.where(label == 0, -1.0, 0.0))  # the paper's coding: 1 flood, -1 dry, 0 unlabeled
    # f runs from -1 (sure it is dry) to +1 (sure it is flood). The paper builds it from two output maps;
    # our model has one sigmoid output, so this is the paper's one-output version (2*sigmoid(s) - 1 = tanh(s/2)).
    f = 2.0 * y_pred - 1.0

    pad = [[0, 0], [1, 1], [1, 1], [0, 0]]  # mirror the edges so every pixel has 8 neighbours, as in the paper
    gt_pad, h_pad = tf.pad(gt, pad, "REFLECT"), tf.pad(h, pad, "REFLECT")
    H, W = tf.shape(gt)[1], tf.shape(gt)[2]

    penalty = tf.zeros_like(gt)
    for dy in range(3):
        for dx in range(3):
            if (dy, dx) == (1, 1):
                continue  # that is the pixel itself
            gt_n = gt_pad[:, dy:dy + H, dx:dx + W]      # the neighbour's label
            dh = h - h_pad[:, dy:dy + H, dx:dx + W]     # eq 1: this pixel's height minus the neighbour's
            # eq 4: the rule only applies if the neighbour is flood and we are not higher, or it is dry and we are not lower
            w = tf.cast(-gt_n * dh >= 0, tf.float32) * tf.abs(gt_n)
            penalty += w * (1.0 - gt_n * f)             # eq 3: 0 when the prediction obeys the rule, up to 2 when it breaks it

    known = tf.abs(gt)  # eq 2 only counts labeled pixels
    return tf.reduce_sum(penalty * known) / tf.maximum(tf.reduce_sum(known), 1.0)

def flood_loss(y_true, y_pred):  # eq 7
    return masked_bce(y_true[..., :1], y_pred) + EVA_LAMBDA * eva_loss(y_true, y_pred)

def add_dem(x, y):
    # the loss needs the elevation, and Keras only hands a loss the labels, so the DEM rides along as a 2nd label channel
    return x, tf.concat([y, x[..., 3:4]], axis=-1)  # channel 3 = elevation in metres (not clipped)

## Build the model: ResNet50 encoder
## Tutorial Repo this code came from:
https://github.com/nikhilroxtomar/Semantic-Segmentation-Architecture/blob/main/TensorFlow/resnet50_unet.py

In [ ]:
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, Conv2DTranspose, Concatenate, Input
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50

def conv_block(x, num_filters):
    x = Conv2D(num_filters, 3, padding="same")(x) #the 3 is how many diff patterns model looks for
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Conv2D(num_filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x

def decoder_block(x, skip_features, num_filters):
    x = Conv2DTranspose(num_filters, 2, strides=2, padding="same")(x)   # learned 2x so 16x16 goes to 32x32
    x = Concatenate()([x, skip_features])
    return conv_block(x, num_filters)

def build_resnet50_unet(input_shape):
    inputs = Input(input_shape)
    resnet50 = ResNet50(include_top=False, weights=None, input_tensor=inputs)

    # ImageNet weights only exist for 3 channels: copy them over, and give conv1 a 4th and 5th (DEM) input
    # that starts at zero, so the model starts out identical to the RGB one
    pretrained = ResNet50(include_top=False, weights="imagenet", input_shape=(256, 256, 3))
    for layer, src in zip(resnet50.layers, pretrained.layers):
        weights = src.get_weights()
        if layer.name == "conv1_conv":
            kernel, bias = weights
            weights = [np.concatenate([kernel, np.zeros_like(kernel[:, :, :2])], axis=2), bias]
        layer.set_weights(weights)

    # Encoder skips (sizes for 256 x 256 tiles)
    s1 = inputs                                          # 256 
    s2 = resnet50.get_layer("conv1_relu").output         # 128
    s3 = resnet50.get_layer("conv2_block3_out").output   # 64
    s4 = resnet50.get_layer("conv3_block4_out").output   # 32

    # Bridge
    b1 = resnet50.get_layer("conv4_block6_out").output   # 16

    # Decoder
    d1 = decoder_block(b1, s4, 512)   # 32
    d2 = decoder_block(d1, s3, 256)   # 64
    d3 = decoder_block(d2, s2, 128)   # 128
    d4 = decoder_block(d3, s1, 64)    # 256

    outputs = Conv2D(1, 1, padding="same", activation="sigmoid")(d4)
    return Model(inputs, outputs, name="ResNet50_U-Net"), resnet50

model, encoder = build_resnet50_unet((256, 256, 5))
print(model.input_shape, "->", model.output_shape)

In [ ]:
from pathlib import Path

#freeze encoder
encoder.trainable = False
encoder.get_layer("conv1_conv").trainable = True  # the only encoder layer that can learn from the DEM

model.compile(
    optimizer=keras.optimizers.Adam(1e-3, clipnorm=1.0),  # clipnorm: one bad batch cannot blow up the weights
    loss=flood_loss,  # masked BCE + the elevation rule (set EVA_LAMBDA = 0 for the old loss)
    jit_compile=False,  # XLA's compiled step needs more than the 2080's 8 GB
)

#save best weights
Path("../outputs").mkdir(exist_ok=True)
checkpoint = keras.callbacks.ModelCheckpoint(
    "../outputs/unet_dem_eva_2dem.weights.h5",
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,
)

MEAN_BGR = tf.constant([103.939, 116.779, 123.68])  # what preprocess_input subtracted (blue, green, red)

def augment(x, y):

    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    x, y = tf.image.rot90(x, k), tf.image.rot90(y, k)
    if tf.random.uniform([]) < 0.5:
        x, y = tf.image.flip_left_right(x), tf.image.flip_left_right(y)

    # Random brightness and contrast on the photo only (not the DEM), so the model does not depend on exposure.
    # The held-out Ida imagery is about 30% darker than Harvey and the model missed nearly all of its flood.
    rgb = x[..., :3] + MEAN_BGR  # undo preprocess_input: back to 0-255
    rgb = rgb * tf.random.uniform([], 0.6, 1.4)
    mean = tf.reduce_mean(rgb)
    rgb = (rgb - mean) * tf.random.uniform([], 0.8, 1.2) + mean
    x = tf.concat([tf.clip_by_value(rgb, 0.0, 255.0) - MEAN_BGR, x[..., 3:]], axis=-1)
    return x, y

# keep the full arrays in RAM; only each batch gets copied to the GPU
with tf.device("/CPU:0"):
    def train_tiles():  # reads tiles straight from the numpy arrays; from_tensor_slices would copy them all
        for i in np.random.permutation(len(X_train_in)):
            yield X_train_in[i], y_train_in[i]

    train_ds = (tf.data.Dataset.from_generator(train_tiles, output_signature=(
                    tf.TensorSpec((256, 256, 5), tf.float32), tf.TensorSpec((256, 256, 1), tf.float32)))
                .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
                .map(add_dem)
                .batch(4).prefetch(2))
    val_ds = tf.data.Dataset.from_tensor_slices((X_val_in, y_val_in)).map(add_dem).batch(4)

#trian
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[checkpoint],
)

In [ ]:
#load best weights
model.load_weights("../outputs/unet_dem_eva_2dem.weights.h5")

probs = model.predict(X_val_in, batch_size=8)[..., 0] #check against y_val

pred_flood = probs > 0.5

known = y_val != -1 #known if not -1 ehich is unknown

true_flood = y_val == 1 #whats actually water
true_dry = y_val == 0 #whats actually dry
pred_dry = ~pred_flood # everywhere the model got the opposite of flood

flood_inter = np.sum(pred_flood & true_flood & known)
flood_union = np.sum((pred_flood | true_flood) & known)
iou_flood = flood_inter / flood_union

dry_inter = np.sum(pred_dry & true_dry & known)
dry_union = np.sum((pred_dry | true_dry) & known)
iou_dry = dry_inter / dry_union

print(f"flood IoU:  {iou_flood:.3f}")
print(f"dry IoU:    {iou_dry:.3f}")

print(f"model says flood: {pred_flood[known].mean():.1%}   label says flood: {true_flood[known].mean():.1%}")



In [ ]:
for cut in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pf = probs > cut
    f = np.sum(pf & true_flood & known) / np.sum((pf | true_flood) & known)
    d = np.sum(~pf & true_dry & known) / np.sum((~pf | true_dry) & known)
    print(f"cutoff {cut}: flood IoU {f:.3f}  dry IoU {d:.3f}  says flood {pf[known].mean():.1%}")

In [ ]:
import matplotlib.pyplot as plt

tiles = [10, 40, 70, 100]   # 4 tiles from image 12 to look at

fig, axes = plt.subplots(len(tiles), 3, figsize=(12, 4 * len(tiles)))
for row, t in enumerate(tiles):
    axes[row, 0].imshow(X_val_rgb[t])                          # the photo
    axes[row, 1].imshow(y_val[t], cmap="Blues", vmin=-1, vmax=1)  # answer
    axes[row, 2].imshow(pred_flood[t], cmap="Blues", vmin=0, vmax=1)           # model's guess
    axes[row, 0].set_title(f"tile {t}: photo")
    axes[row, 1].set_title("label (white = unknown)")
    axes[row, 2].set_title("prediction (blue = flood)")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
CUTOFF = 0.4


model.load_weights("../outputs/unet_dem_eva_2dem.weights.h5")

#predict on test
test_probs = model.predict(X_test_in, batch_size=8)[..., 0]


test_pred_flood = test_probs > CUTOFF
test_pred_dry = ~test_pred_flood


test_known = y_test != -1
test_true_flood = y_test == 1
test_true_dry = y_test == 0

test_iou_flood = np.sum(test_pred_flood & test_true_flood & test_known) / np.sum((test_pred_flood | test_true_flood) & test_known)
test_iou_dry = np.sum(test_pred_dry & test_true_dry & test_known) / np.sum((test_pred_dry | test_true_dry) & test_known)

print(f"TEST (cutoff {CUTOFF})")
print(f"flood IoU: {test_iou_flood:.3f}")
print(f"dry IoU:   {test_iou_dry:.3f}")
print(f"model says flood: {test_pred_flood[test_known].mean():.1%}   label says flood: {test_true_flood[test_known].mean():.1%}")

In [ ]:
for i in SPLITS["test"]:
    Xi, yi = get_tiles([i])
    pi = model.predict(preprocess(Xi), batch_size=8, verbose=0)[..., 0] > CUTOFF
    k, tf_, td = yi != -1, yi == 1, yi == 0
    f = np.sum(pi & tf_ & k) / np.sum((pi | tf_) & k)
    d = np.sum(~pi & td & k) / np.sum((~pi | td) & k)
    print(f"image {i}: flood IoU {f:.3f}  dry IoU {d:.3f}  says flood {pi[k].mean():.1%}  label {tf_[k].mean():.1%}")

In [ ]:
# Held-out hard case: Hurricane Ida. The model never trained on this storm, and it is not used to pick the cutoff.
# Flood IoU only: this image has too few dry labels for a dry score to mean much.
for i in SPLITS["ida_hard_case"]:
    Xi, yi = get_tiles([i])
    pi = model.predict(preprocess(Xi), batch_size=8, verbose=0)[..., 0] > CUTOFF
    k, tf_ = yi != -1, yi == 1
    f = np.sum(pi & tf_ & k) / np.sum((pi | tf_) & k)
    print(f"IDA image {i}: flood IoU {f:.3f}  says flood {pi[k].mean():.1%}  label {tf_[k].mean():.1%}")


In [ ]:
# New-storm test: 2019 Arkansas River flood. Never trained on, and not used to pick the epoch or the cutoff.
# Only look at this when reporting the final model; using it to choose between models would make it a second validation set.
for i in SPLITS["new_storm_test"]:
    Xi, yi = get_tiles([i])
    pi = model.predict(preprocess(Xi), batch_size=8, verbose=0)[..., 0] > CUTOFF
    k, tf_ = yi != -1, yi == 1
    tp, fp, fn = np.sum(pi & tf_ & k), np.sum(pi & ~tf_ & k), np.sum(~pi & tf_ & k)
    flood = tp / (tp + fp + fn)
    dry = np.sum(~pi & ~tf_ & k) / np.sum((~pi | ~tf_) & k)
    precision, recall = tp / (tp + fp), tp / (tp + fn)
    print(f"ARKANSAS image {i}: flood IoU {flood:.3f}  dry IoU {dry:.3f}  precision {precision:.3f}  recall {recall:.3f}  "
          f"F1 {2 * precision * recall / (precision + recall):.3f}  says flood {pi[k].mean():.1%}  label {tf_[k].mean():.1%}")


In [ ]:
tiles = [20, 70, 120, 170, 220, 270]   
fig, axes = plt.subplots(len(tiles), 3, figsize=(12, 4 * len(tiles)))
for row, t in enumerate(tiles):
    axes[row, 0].imshow(X_test_rgb[t])                             
    axes[row, 1].imshow(y_test[t], cmap="Blues", vmin=-1, vmax=1)  
    axes[row, 2].imshow(test_pred_flood[t], cmap="Blues", vmin=0, vmax=1)          
    axes[row, 0].set_title(f"test tile {t}: photo")
    axes[row, 1].set_title("label (white = unknown)")
    axes[row, 2].set_title(f"prediction, cutoff {CUTOFF}")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()